# Random Forest with ECG — Charis

This compares PPG alone with PPG+ECG using the same training and validation windows. The RF settings, pressure labels and recording-overlap folds stay fixed. ECG is the change being tested.

Training: 11,291 windows from 1,308 recordings. Validation: 857 windows from 100 recordings. The old test is not used. [Results and explanation](RF_ECG_RESULTS.md).

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
RUN=Path('outputs/rf_ecg_nov1')
metrics=pd.read_csv(RUN/'metrics.csv')
validation=metrics[metrics.split.eq('validation')]
display(validation.pivot(index=['model','modality'],columns='target',values=['mae_mmHg','rmse_mmHg']).round(2))

mae_mmHg        rmse_mmHg       
target                         dbp    sbp       dbp    sbp
model           modality                                  
Random Forest   PPG           8.34  15.76     11.44  21.23
                PPG + ECG     7.63  15.03     10.92  20.43
Training mean   Constant     10.02  18.70     13.17  23.44
Training median Constant     10.17  18.96     13.79  23.89

## Result

RF with ECG reached MAE of 15.03 mmHg systolic and 7.63 diastolic, compared with 15.76/8.34 for PPG alone. Both RMSE values improved too. This is a matched development comparison, not a new final-test score.

In [2]:
display(validation[['model','modality','target','within_5_mmHg_pct','within_10_mmHg_pct','within_15_mmHg_pct']].round(1))

,model,modality,target,within_5_mmHg_pct,within_10_mmHg_pct,within_15_mmHg_pct
2,Random Forest,PPG,sbp,26.4,44.8,60.2
3,Random Forest,PPG,dbp,41.4,69.5,86.2
6,Random Forest,PPG + ECG,sbp,26.6,49.0,64.2
7,Random Forest,PPG + ECG,dbp,44.0,77.8,88.7
8,Training mean,Constant,sbp,16.9,33.3,49.1
9,Training mean,Constant,dbp,27.4,58.8,84.6
10,Training median,Constant,sbp,15.6,32.3,50.8
11,Training median,Constant,dbp,29.6,62.4,80.7


## What “within 10” means

77.8% of RF+ECG diastolic predictions had absolute error <=10 mmHg. That is error coverage, not classification accuracy. Report it alongside MAE and RMSE. The percentages use the same 857 scored windows for both inputs.

In [3]:
display(pd.read_csv(RUN/'paired_ecg_gains.csv').round(3))
display(pd.read_json(RUN/'ecg_overlap_summary.json',typ='series'))
display(pd.read_csv(RUN/'rf_ecg_feature_importance.csv').head(12).round(4))

,model,target,group_macro_mae_gain_from_ecg,ci_low,ci_high
0,Random Forest,sbp,0.744,-0.260,1.705
1,Random Forest,dbp,0.704,0.192,1.223


exact_match_events                                                           2
training_windows_excluded                                                    0
validation_windows                                                         857
training_windows                                                         11291
groups_merged                                                                0
limitation                   Exact nonconstant ECG full records and selecte...
dtype: object

,feature,training_split_importance
0,ecg_high_power_fraction,0.0405
1,ecg_raw_mean,0.0395
2,ecg_z_min,0.0394
3,ecg_raw_p05,0.0377
4,ecg_z_iqr,0.0313
5,ecg_low_power_fraction,0.0294
6,ecg_z_median,0.0271
7,pulse_upslope_median,0.0267
8,vpg_skew,0.0261
9,ecg_raw_p95,0.0225


## Limits and next steps

Recording IDs do not prove patient independence. The labels remain provisional, and extreme pressures remain hard. The validation set has been inspected repeatedly, so the chosen model still needs an untouched final evaluation. Tree split importance is descriptive, not causal or a percentage of accuracy.

`train_rf_ecg.py` recreates the RF comparison from the saved feature tables and folds. Use a fresh output directory. `predict_ppg_ecg.py` runs a regenerated joint model on synchronized 125 Hz PPG/ECG arrays. No model binaries or raw waveforms are committed.